# Laboratorio 13 — Modelos de Difusión
**Redes Neuronales y Aprendizaje Profundo — UNI FIIS, Unidad de Posgrado**
Aldo Camargo Fernández-Baca, Ph.D.

Acompaña a las instrucciones `Lab13_Difusion.pdf`. Usted implementará las piezas centrales de un modelo de difusión — el proceso directo, la pérdida de DDPM, los muestreadores DDPM y DDIM, y la guía sin clasificador — y las usará sobre datos 2D y sobre **Fashion-MNIST**.

| Parte | Contenido | Tiempo | Puntos |
|---|---|---|---|
| 0 | Preparación | 10 min | — |
| 1 | Proceso directo, posterior y horarios | 15 min | 3 |
| 2 | Pérdida y muestreo DDPM; 8 gaussianas | 25 min | 5 |
| 3 | DDIM sobre Fashion-MNIST: pasos vs. calidad | 30 min | 5 |
| 4 | Classifier-free guidance: fidelidad vs. diversidad | 30 min | 5 |
| 5 | Preguntas de cierre | 10 min | 2 |

**Reglas del notebook**
* Complete solo las celdas marcadas con `# TODO`. Cada función tiene una **prueba automática** debajo: no avance hasta que imprima `OK`.
* Use **GPU** (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*).
* Convención de índices (igual que en la clase): el índice de Python `t = 0, …, T-1` corresponde al paso $t+1$ de las notas.
* Las celdas de entrenamiento tardan ≈4 min cada una con T4. Mientras corren, Colab le deja **escribir** en otras celdas (se ejecutarán en cola): aproveche ese tiempo.

## Parte 0. Preparación (10 min)

Ejecute las celdas de esta sección sin modificarlas. Definen las utilidades, los horarios, las redes de la clase (`MLPRuido`, `UNet`), el conjunto Fashion-MNIST y un **clasificador auxiliar** que usaremos para medir la calidad de las muestras con un FID simplificado (FID\*), como en la sección G8 de la Clase 11.

In [ ]:
import math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from scipy import linalg

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| dispositivo:', device)
if device == 'cpu':
    print('ATENCIÓN: sin GPU las Partes 3 y 4 serán muy lentas. Cambie el tipo de entorno.')

def sincronizar():
    if device == 'cuda': torch.cuda.synchronize()

def mostrar(imgs, nrow=10, titulo='', ax=None):
    grid = make_grid(imgs.detach().cpu().float().clamp(-1, 1) * 0.5 + 0.5, nrow=nrow, padding=2, pad_value=1)
    ax = ax or plt.figure(figsize=(nrow * 0.6, (len(imgs) // nrow) * 0.6 + 0.4)).gca()
    ax.imshow(grid.permute(1, 2, 0)); ax.axis('off'); ax.set_title(titulo)

def horario_lineal(T, b1=1e-4, bT=0.02):
    return torch.linspace(b1, bT, T, dtype=torch.float64)

def horario_coseno(T, s=0.008):
    t = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos(((t / T + s) / (1 + s)) * math.pi / 2) ** 2
    abar = f / f[0]
    return (1 - abar[1:] / abar[:-1]).clamp(max=0.999)

def e(v, t, x):
    # toma v[t] (un valor por elemento del lote) y lo deja listo para multiplicar a x
    return v[t].view(-1, *([1] * (x.dim() - 1)))

def rapido(modelo):
    # envoltura para muestrear en media precisión en GPU (mucho más rápido en T4); devuelve float32
    def f(x, t, **kw):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
            return modelo(x, t, **kw).float()
    return f

T = 1000

In [ ]:
class EmbTiempo(nn.Module):
    # embedding sinusoidal de Vaswani et al. (2017), usado por DDPM para el paso t
    def __init__(self, d):
        super().__init__(); self.d = d
    def forward(self, t):
        mitad = self.d // 2
        frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
        ang = t.float().unsqueeze(-1) * frec
        return torch.cat([ang.sin(), ang.cos()], -1)

class MLPRuido(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Sequential(EmbTiempo(64), nn.Linear(64, h), nn.SiLU())
        self.l1, self.l2, self.l3 = nn.Linear(2, h), nn.Linear(h, h), nn.Linear(h, h)
        self.out = nn.Linear(h, 2)
    def forward(self, x, t):
        e = self.emb(t)
        h = F.silu(self.l1(x) + e)
        h = F.silu(self.l2(h) + e)
        h = F.silu(self.l3(h) + e)
        return self.out(h)

class BloqueRes(nn.Module):
    def __init__(self, c_in, c_out, d_emb, dropout=0.1):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, c_in), nn.Conv2d(c_in, c_out, 3, padding=1)
        self.emb = nn.Linear(d_emb, c_out)
        self.n2, self.c2 = nn.GroupNorm(8, c_out), nn.Conv2d(c_out, c_out, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.salto = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()
    def forward(self, x, emb):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.emb(emb)[:, :, None, None]              # el paso t (y la clase) entra aquí
        h = self.c2(self.drop(F.silu(self.n2(h))))
        return h + self.salto(x)

class Atencion(nn.Module):
    def __init__(self, c, cabezas=4):
        super().__init__()
        self.norm, self.att = nn.GroupNorm(8, c), nn.MultiheadAttention(c, cabezas, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).flatten(2).transpose(1, 2)          # (B, H*W, C): cada píxel es un token
        h, _ = self.att(h, h, h)
        return x + h.transpose(1, 2).view(B, C, H, W)

class UNet(nn.Module):
    def __init__(self, c=64, n_clases=None):
        super().__init__()
        d = 4 * c
        self.emb_t = nn.Sequential(EmbTiempo(c), nn.Linear(c, d), nn.SiLU(), nn.Linear(d, d))
        self.n_clases = n_clases
        self.emb_y = nn.Embedding(n_clases + 1, d) if n_clases else None   # índice n_clases = "vacío" (D8)
        self.entrada = nn.Conv2d(1, c, 3, padding=1)
        self.e1 = BloqueRes(c, c, d);          self.baja1 = nn.Conv2d(c, c, 3, 2, 1)           # 28 -> 14
        self.e2 = BloqueRes(c, 2 * c, d);      self.baja2 = nn.Conv2d(2 * c, 2 * c, 3, 2, 1)   # 14 -> 7
        self.m1, self.att, self.m2 = BloqueRes(2 * c, 2 * c, d), Atencion(2 * c), BloqueRes(2 * c, 2 * c, d)
        self.sube2 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(2 * c, 2 * c, 3, padding=1))  # 7 -> 14
        self.d2 = BloqueRes(4 * c, 2 * c, d)
        self.sube1 = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(2 * c, c, 3, padding=1))      # 14 -> 28
        self.d1 = BloqueRes(2 * c, c, d)
        self.salida = nn.Sequential(nn.GroupNorm(8, c), nn.SiLU(), nn.Conv2d(c, 1, 3, padding=1))
    def forward(self, x, t, y=None):
        emb = self.emb_t(t)
        if self.emb_y is not None:
            if y is None: y = torch.full((len(x),), self.n_clases, device=x.device)
            emb = emb + self.emb_y(y)
        h0 = self.entrada(x)
        h1 = self.e1(h0, emb)                                 # 28x28
        h2 = self.e2(self.baja1(h1), emb)                     # 14x14
        h = self.m2(self.att(self.m1(self.baja2(h2), emb)), emb)   # 7x7
        h = self.d2(torch.cat([self.sube2(h), h2], 1), emb)   # 14x14
        h = self.d1(torch.cat([self.sube1(h), h1], 1), emb)   # 28x28
        return self.salida(h)

print('MLPRuido y UNet definidas (idénticas al notebook de la Clase 13).')

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])   # [0,1] -> [-1,1]
fm_train = torchvision.datasets.FashionMNIST('data', train=True, download=True, transform=transform)
fm_test  = torchvision.datasets.FashionMNIST('data', train=False, download=True, transform=transform)
CLASES = ['camiseta', 'pantalón', 'suéter', 'vestido', 'abrigo', 'sandalia', 'camisa', 'zapatilla', 'bolso', 'botín']
loader = torch.utils.data.DataLoader(fm_train, batch_size=128, shuffle=True, drop_last=True, num_workers=2)
x_ej, y_ej = next(iter(loader))
mostrar(x_ej[:30], titulo='Fashion-MNIST: ' + ', '.join(CLASES[i] for i in y_ej[:10].tolist()) + ', …'); plt.show()

In [ ]:
# ---- Clasificador auxiliar y FID* (como en la Clase 11, G8) -------------------------------
class Clasificador(nn.Module):
    def __init__(self):
        super().__init__()
        self.caracteristicas = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.salida = nn.Linear(128, 10)
    def forward(self, x): return self.salida(self.caracteristicas(x))

torch.manual_seed(0)
clf = Clasificador().to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
for ep in range(3):
    for x, y in loader:
        loss = F.cross_entropy(clf(x.to(device)), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()
test_loader = torch.utils.data.DataLoader(fm_test, batch_size=500)
with torch.no_grad():
    acc = np.mean([(clf(x.to(device)).argmax(1).cpu() == y).float().mean().item() for x, y in test_loader])
print(f'Exactitud del clasificador auxiliar en prueba: {acc:.3f}')

@torch.no_grad()
def caracteristicas(imgs, lote=500):
    feats, clases = [], []
    for i in range(0, len(imgs), lote):
        f = clf.caracteristicas(imgs[i:i + lote].to(device).float().clamp(-1, 1))   # recorte al rango de los datos
        feats.append(f.cpu()); clases.append(clf.salida(f).argmax(1).cpu())
    return torch.cat(feats).double().numpy(), torch.cat(clases)

def fid(f1, f2):
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    raiz = linalg.sqrtm(s1 @ s2)
    raiz = (raiz[0] if isinstance(raiz, tuple) else raiz).real
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1 + s2 - 2 * raiz))

def diversidad_por_clase(f, clases_pedidas):
    # distancia media entre pares de características dentro de cada clase pedida, promediada sobre clases
    d = []
    for c in clases_pedidas.unique():
        fc = torch.tensor(f[(clases_pedidas == c).numpy()])
        if len(fc) > 1: d.append(torch.pdist(fc).mean().item())
    return float(np.mean(d))

N_FID = 1000
reales = torch.stack([fm_test[i][0] for i in range(N_FID)])
y_reales = torch.tensor([fm_test[i][1] for i in range(N_FID)])
f_reales, _ = caracteristicas(reales)
f_ref, _ = caracteristicas(torch.stack([fm_test[i][0] for i in range(N_FID, 2 * N_FID)]))
DIV_REAL = diversidad_por_clase(f_reales, y_reales)
print(f'FID* entre dos mitades del conjunto de prueba (piso por ruido muestral): {fid(f_reales, f_ref):.2f}')
print(f'Diversidad intra-clase de los datos reales: {DIV_REAL:.2f}')

## Parte 1. Proceso directo, posterior y horarios (15 min, 3 pts)

La clase `Horario` ya calcula $\beta_t$, $\alpha_t=1-\beta_t$, $\bar\alpha_t$ y $\bar\alpha_{t-1}$ (con $\bar\alpha_0=1$). Usted completa tres funciones.

In [ ]:
class Horario:
    def __init__(self, betas):
        self.T = len(betas)
        self.betas = betas.float().to(device)
        self.alphas = 1 - self.betas
        self.abar = torch.cumprod(self.alphas, 0)
        self.abar_prev = torch.cat([torch.ones(1, device=device), self.abar[:-1]])

h_lin, h_cos = Horario(horario_lineal(T)), Horario(horario_coseno(T))

### Tarea 1.1 — Forma cerrada (1 pt)
Implemente $\mathbf{x}_t=\sqrt{\bar\alpha_t}\,\mathbf{x}_0+\sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon$ (ec. 4 de las notas). Use la función `e(v, t, x)` para que funcione con lotes de cualquier forma.

In [ ]:
def q_sample(h, x0, t, eps):
    # x0, eps: (B, ...);  t: (B,) enteros en [0, T-1]
    # TODO 1.1
    raise NotImplementedError

# ---- prueba: comparamos con aplicar el paso x_t = sqrt(1-b) x_{t-1} + sqrt(b) eps, 300 veces
torch.manual_seed(0)
x0 = torch.tensor([1.5, -0.5], device=device).repeat(200000, 1)
x = x0.clone()
for i in range(300):
    x = (1 - h_lin.betas[i]).sqrt() * x + h_lin.betas[i].sqrt() * torch.randn_like(x)
xc = q_sample(h_lin, x0, torch.full((len(x0),), 299, device=device), torch.randn_like(x0))
assert xc.shape == x0.shape, 'forma incorrecta'
assert torch.allclose(x.mean(0), xc.mean(0), atol=0.01) and torch.allclose(x.var(0), xc.var(0), atol=0.01), 'no coincide con la cadena iterada'
print('OK  media iterada', x.mean(0).cpu().numpy().round(3), ' cerrada', xc.mean(0).cpu().numpy().round(3))

### Tarea 1.2 — Posterior (1 pt)
Implemente la media y la varianza del posterior $q(\mathbf{x}_{t-1}\mid\mathbf{x}_t,\mathbf{x}_0)$ (ec. 9 de las notas):
$$\tilde{\boldsymbol\mu}_t=\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,\mathbf{x}_0+\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,\mathbf{x}_t,\qquad \tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t .$$
La prueba verifica la identidad de la ec. 10, $\tilde{\boldsymbol\mu}_t=\frac{1}{\sqrt{\alpha_t}}\big(\mathbf{x}_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\boldsymbol\epsilon\big)$.

In [ ]:
def posterior(h, x0, xt, t):
    # devuelve (mu_til, beta_til), ambos con la forma de xt (beta_til se puede devolver con broadcast)
    # TODO 1.2
    raise NotImplementedError

# ---- prueba
torch.manual_seed(1)
x0 = torch.randn(64, 1, 28, 28, device=device); eps = torch.randn_like(x0)
t = torch.randint(1, T, (64,), device=device)
xt = q_sample(h_lin, x0, t, eps)
mu, bt = posterior(h_lin, x0, xt, t)
mu_eps = (xt - e(h_lin.betas, t, xt) / (1 - e(h_lin.abar, t, xt)).sqrt() * eps) / e(h_lin.alphas, t, xt).sqrt()
assert torch.allclose(mu, mu_eps, atol=1e-3), 'la media no satisface la ec. 10'
assert (bt <= e(h_lin.betas, t, xt) + 1e-8).all(), 'beta_til debería ser <= beta'
print('OK  máx |mu_til - mu(eps)| =', (mu - mu_eps).abs().max().item())

### Tarea 1.3 — Horarios (1 pt)
Para los horarios lineal y coseno calcule: (a) el paso $t$ (en la numeración de las notas, $1..T$) donde $\mathrm{SNR}(t)=\bar\alpha_t/(1-\bar\alpha_t)$ cruza 1; (b) cuántos pasos tienen $\mathrm{SNR}(t)<10^{-2}$ (la imagen es prácticamente ruido). Grafique $\log_{10}\mathrm{SNR}$ para ambos.

In [ ]:
def analizar_horario(h):
    # devuelve (t_cruce, n_pasos_ruido): t_cruce en numeración 1..T
    # TODO 1.3
    raise NotImplementedError

for nombre, h in [('lineal', h_lin), ('coseno', h_cos)]:
    tc, nr = analizar_horario(h)
    print(f'{nombre:7s}: SNR cruza 1 en t = {tc:4d} | pasos con SNR < 0.01: {nr}')
    plt.plot(range(1, T + 1), torch.log10(h.abar / (1 - h.abar)).cpu(), label=nombre)
plt.axhline(0, color='gray', ls=':'); plt.axhline(-2, color='gray', ls='--')
plt.xlabel('t'); plt.ylabel(r'$\log_{10}\,\mathrm{SNR}(t)$'); plt.legend(); plt.grid(alpha=.3); plt.show()

**Respuesta:** _(escriba aquí)_

## Parte 2. Pérdida y muestreo DDPM sobre 8 gaussianas (25 min, 5 pts)

Usamos los mismos datos que provocaron el **colapso de modos** en la Clase 11 (G4): 8 gaussianas pequeñas en un anillo de radio 2.

In [ ]:
K_MODOS, RADIO, S_MODO = 8, 2.0, 0.05
centros = torch.tensor([[RADIO * math.cos(2 * math.pi * i / K_MODOS), RADIO * math.sin(2 * math.pi * i / K_MODOS)]
                        for i in range(K_MODOS)])
def muestrear_anillo(n):
    return centros[torch.randint(0, K_MODOS, (n,))] + S_MODO * torch.randn(n, 2)

def evaluar_anillo(x):
    # (modos cubiertos con >= 2 % de las muestras dentro de 3 sigma, fracción de muestras dentro de algún modo)
    dist = torch.cdist(x.cpu(), centros)
    cerca = dist.min(1).values < 3 * S_MODO
    frac = torch.bincount(dist.argmin(1)[cerca], minlength=K_MODOS).float() / len(x)
    return int((frac > 0.02).sum()), cerca.float().mean().item()

def oraculo_gaussiano(h, m, s):
    # epsilon EXACTO para datos x0 ~ N(m, s^2 I): q(x_t) = N(sqrt(abar) m, (abar s^2 + 1 - abar) I)
    def eps(x, t, **kw):
        ab = e(h.abar, t, x)
        return (1 - ab).sqrt() * (x - ab.sqrt() * m) / (ab * s ** 2 + 1 - ab)
    return eps

### Tarea 2.1 — La pérdida simple (1 pt)
Implemente el **Algoritmo 1** (ec. 16): muestree $t\sim\mathcal U\{0,\dots,T-1\}$ y $\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,\mathbf I)$ por ejemplo del lote, forme $\mathbf{x}_t$ con `q_sample` y devuelva el error cuadrático medio entre $\boldsymbol\epsilon$ y `modelo(x_t, t, **kw)`. Los `**kw` (la etiqueta `y`) se usarán en la Parte 4.

In [ ]:
def perdida_simple(modelo, h, x0, **kw):
    # TODO 2.1
    raise NotImplementedError

# ---- prueba: con un modelo que siempre predice 0, la pérdida debe ser E[eps^2] = 1
cero = lambda x, t, **kw: torch.zeros_like(x)
l0 = perdida_simple(cero, h_lin, torch.randn(4096, 2, device=device))
assert abs(l0.item() - 1) < 0.05, f'esperaba ~1, obtuve {l0.item():.3f}'
# ---- con el oráculo exacto (el mejor modelo posible) la pérdida NO es cero: alcanza el mínimo teórico
#      E||eps - E[eps|x_t]||^2 / d = promedio_t  abar s^2 / (abar s^2 + 1 - abar)
m0 = torch.tensor([2.0, -1.0], device=device)
torch.manual_seed(0)
l1 = torch.stack([perdida_simple(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, m0 + 0.5 * torch.randn(4096, 2, device=device))
                  for _ in range(20)]).mean()
minimo = (h_lin.abar * 0.25 / (h_lin.abar * 0.25 + 1 - h_lin.abar)).mean()
assert abs(l1.item() - minimo.item()) < 0.01, f'con el oráculo esperaba {minimo.item():.3f}, obtuve {l1.item():.3f}'
print(f'OK  modelo cero: {l0.item():.3f}   oráculo: {l1.item():.3f}   mínimo teórico: {minimo.item():.3f}')

### Tarea 2.2 — Muestreo DDPM (1.5 pts)
Implemente el **Algoritmo 2**: desde $\mathbf{x}_T\sim\mathcal N(\mathbf 0,\mathbf I)$ (o `xT` si se da), para $t=T-1,\dots,0$:
$$\mathbf{x}_{t-1}=\frac{1}{\sqrt{\alpha_t}}\Big(\mathbf{x}_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\boldsymbol\epsilon_\theta(\mathbf{x}_t,t)\Big)+\sigma_t\mathbf z,$$
sin ruido en el último paso. Con `sigma='beta'` use $\sigma_t^2=\beta_t$; con `sigma='btil'` use $\sigma_t^2=\tilde\beta_t$ (puede reutilizar `posterior` o calcularlo directamente). Recuerde que `t` que recibe el modelo es un **tensor** de tamaño $B$.

La prueba usa el oráculo exacto para datos $\mathcal N(\mathbf m, 0.5^2\mathbf I)$: si su muestreador es correcto, las muestras deben tener esa media y esa desviación.

In [ ]:
@torch.no_grad()
def muestrear_ddpm(modelo, h, forma, sigma='beta', xT=None, **kw):
    # TODO 2.2
    raise NotImplementedError

# ---- prueba con el oráculo gaussiano
torch.manual_seed(2)
for sg in ['beta', 'btil']:
    xs = muestrear_ddpm(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, (20000, 2), sigma=sg)
    print(f'sigma={sg:4s}: media {xs.mean(0).cpu().numpy().round(3)}  desv. {xs.std(0).cpu().numpy().round(3)}   (esperado [2, -1] y 0.5)')
    assert torch.allclose(xs.mean(0), m0, atol=0.03) and torch.allclose(xs.std(0), torch.tensor(0.5, device=device), atol=0.03)
print('OK')

### Tarea 2.3 — ¿Colapsa un modelo de difusión? (1.5 pts)
La celda entrena `MLPRuido` con **su** pérdida sobre el anillo de 8 gaussianas (≈1 min) y muestrea 3000 puntos con **su** muestreador. Complete la figura: datos en gris, muestras en color, y en el título los modos cubiertos y la fracción de muestras dentro de algún modo.

In [ ]:
torch.manual_seed(0)
mlp_anillo = MLPRuido().to(device)
opt = torch.optim.Adam(mlp_anillo.parameters(), lr=2e-3)
PASOS_2D = 6000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, PASOS_2D)
t0 = time.time()
for paso in range(PASOS_2D):
    loss = perdida_simple(mlp_anillo, h_lin, muestrear_anillo(1024).to(device))
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
print(f'entrenado en {time.time()-t0:.0f}s, pérdida final {loss.item():.3f}')

torch.manual_seed(3)
x_beta = muestrear_ddpm(mlp_anillo, h_lin, (3000, 2), sigma='beta').cpu()
real = muestrear_anillo(3000)
# TODO 2.3: figura con datos (gris) y muestras; use evaluar_anillo(x_beta) para el título

**Respuesta:** _(escriba aquí)_

### Tarea 2.4 — $\sigma_t^2=\beta_t$ vs. $\sigma_t^2=\tilde\beta_t$ (1 pt)
Muestree el mismo modelo con `sigma='btil'` (misma semilla) y compare modos cubiertos, fracción en modos y la **desviación estándar de las muestras alrededor de su centro más cercano** (compárela con `S_MODO = 0.05`). ¿Coincide con lo que dicen Ho et al. (“resultados similares”)? ¿Cuál deja las muestras más dispersas y por qué ($\tilde\beta_t$ vs. $\beta_t$)?

In [ ]:
# TODO 2.4

**Respuesta:** _(escriba aquí)_

## Parte 3. DDIM sobre Fashion-MNIST (30 min, 5 pts)

**Primero lance la celda de entrenamiento** (≈4 min con T4). Mientras corre, escriba `muestrear_ddim` en la celda siguiente.

In [ ]:
def entrenar(modelo, h, epocas, lr=2e-4, cond=False, p_vacio=0.1, ema_decay=0.999):
    # Algoritmo 1 con SU perdida_simple, precisión mixta y EMA de los pesos (devuelve el modelo EMA)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    escala = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
    ema = copy.deepcopy(modelo).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    for ep in range(1, epocas + 1):
        t0, perdidas = time.time(), []
        modelo.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            kw = {'y': reemplazar_por_vacio(y, p_vacio, modelo.n_clases)} if cond else {}
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                loss = perdida_simple(modelo, h, x, **kw)
            opt.zero_grad(); escala.scale(loss).backward()
            escala.unscale_(opt); nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            escala.step(opt); escala.update()
            with torch.no_grad():
                for pe, pm in zip(ema.parameters(), modelo.parameters()):
                    pe.mul_(ema_decay).add_(pm, alpha=1 - ema_decay)
            perdidas.append(loss.item())
        print(f'época {ep}/{epocas} | L_simple {np.mean(perdidas):.4f} | {time.time()-t0:.0f}s')
    return ema

EPOCAS = 5
torch.manual_seed(0)
unet = UNet().to(device)
unet_ema = entrenar(unet, h_lin, EPOCAS)

### Tarea 3.1 — DDIM (1.5 pts)
Implemente la ec. 26 de las notas sobre la subsecuencia `seq` de `pasos` índices (ya calculada, de $T-1$ a $0$):
$$\hat{\mathbf{x}}_0=\frac{\mathbf{x}_t-\sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon_\theta}{\sqrt{\bar\alpha_t}},\qquad
\mathbf{x}_{\text{prev}}=\sqrt{\bar\alpha_{\text{prev}}}\,\hat{\mathbf{x}}_0+\sqrt{1-\bar\alpha_{\text{prev}}-\sigma_t^2}\;\boldsymbol\epsilon_\theta+\sigma_t\mathbf z,\qquad
\sigma_t=\eta\sqrt{\tfrac{1-\bar\alpha_{\text{prev}}}{1-\bar\alpha_t}}\sqrt{1-\tfrac{\bar\alpha_t}{\bar\alpha_{\text{prev}}}},$$
donde “prev” es el **siguiente índice de la subsecuencia** y, después del último, $\bar\alpha_{\text{prev}}=1$.

In [ ]:
@torch.no_grad()
def muestrear_ddim(modelo, h, forma, pasos=50, eta=0.0, xT=None, **kw):
    seq = torch.linspace(h.T - 1, 0, pasos).round().long().tolist()
    x = torch.randn(forma, device=device) if xT is None else xT.clone()
    # TODO 3.1
    raise NotImplementedError

# ---- pruebas: con el oráculo y S = T = 1000 pasos debe recuperar N(m, 0.5^2) con eta = 0 y eta = 1
torch.manual_seed(4)
for eta in [0.0, 1.0]:
    xs = muestrear_ddim(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, (20000, 2), pasos=1000, eta=eta)
    print(f'S=1000, eta={eta}: media {xs.mean(0).cpu().numpy().round(3)}  desv. {xs.std(0).cpu().numpy().round(3)}')
    assert torch.allclose(xs.mean(0), m0, atol=0.03) and torch.allclose(xs.std(0), torch.tensor(0.5, device=device), atol=0.03)
# ---- con pocos pasos la media se mantiene, pero la desviación se ENCOGE (error de discretización): obsérvelo
for S_ in [100, 50, 20]:
    for eta in [0.0, 1.0]:
        xs = muestrear_ddim(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, (20000, 2), pasos=S_, eta=eta)
        print(f'S={S_:4d}, eta={eta}: media {xs.mean(0).cpu().numpy().round(3)}  desv. {xs.std(0).mean().item():.3f}')
        assert torch.allclose(xs.mean(0), m0, atol=0.05) and 0.3 < xs.std(0).mean().item() < 0.52
xT = torch.randn(5, 2, device=device)
a = muestrear_ddim(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, (5, 2), pasos=20, xT=xT)
b = muestrear_ddim(oraculo_gaussiano(h_lin, m0, 0.5), h_lin, (5, 2), pasos=20, xT=xT)
assert torch.equal(a, b), 'con eta=0 el muestreo debe ser determinista'
print('OK')

**Observe** la salida de la prueba: incluso con el ruido **exacto**, con pocos pasos la desviación de las muestras queda por debajo de 0.5, y más con $\eta=1$. Es error de discretización, no de la red: los pasos grandes “tiran” las muestras hacia $\hat{\mathbf{x}}_0$, que es una predicción promedio. En imágenes esto se ve como muestras más suaves y menos variadas. Úselo en el análisis de la Tarea 3.2.

### Tarea 3.2 — Pasos vs. calidad vs. tiempo (2 pts)
Con el modelo EMA (`rapido(unet_ema)`), genere `N_FID = 1000` imágenes con cada configuración de la tabla, mida el tiempo de muestreo y el FID\* contra `f_reales`. Complete la lista `filas` y muestre: (a) la tabla, (b) una figura de FID\* vs. número de evaluaciones de la red (eje $x$ logarítmico), (c) 20 muestras de cada configuración. **Use la misma semilla** antes de cada configuración. Copie la tabla en su informe (Tabla 1 de las instrucciones).

*Tiempo esperado con T4: ≈3 min para DDPM con 1000 pasos; segundos para el resto.*

In [ ]:
modelo_u = rapido(unet_ema)
configs = [('DDPM σ²=β', 1000, None), ('DDIM η=0', 10, 0.0), ('DDIM η=0', 25, 0.0), ('DDIM η=0', 50, 0.0),
           ('DDIM η=0', 100, 0.0), ('DDIM η=1', 50, 1.0)]

def generar(f, n=N_FID, lote=500):
    # llama a f(n_lote) por lotes y concatena (para no agotar la memoria)
    return torch.cat([f(min(lote, n - i)).cpu() for i in range(0, n, lote)])

filas, muestras = [], {}
# TODO 3.2: para cada configuración: torch.manual_seed(0); sincronizar(); medir tiempo;
#           x = generar(lambda b: muestrear_ddpm(...)  o  muestrear_ddim(...));
#           f, _ = caracteristicas(x); filas.append((nombre, S_, segundos, fid(f_reales, f)))
#           luego la tabla, la figura FID* vs. pasos y las cuadrículas de muestras

**Respuesta:** _(escriba aquí)_

### Tarea 3.3 — Determinismo e interpolación (1.5 pts)
(a) Genere dos veces con DDIM ($\eta=0$, 50 pasos) desde el mismo `xT` y verifique que las imágenes son idénticas; repita con DDPM y muestre la diferencia máxima.
(b) Implemente la **interpolación esférica** `slerp(z1, z2, lam)` (los ruidos gaussianos en alta dimensión viven cerca de una esfera de radio $\sqrt d$; la interpolación lineal pasa por el interior, donde casi no hay ruido “típico”):
$$\mathrm{slerp}(\mathbf z_1,\mathbf z_2,\lambda)=\frac{\sin((1-\lambda)\theta)}{\sin\theta}\mathbf z_1+\frac{\sin(\lambda\theta)}{\sin\theta}\mathbf z_2,\qquad \cos\theta=\frac{\mathbf z_1\cdot\mathbf z_2}{\|\mathbf z_1\|\,\|\mathbf z_2\|}.$$
Genere 4 filas de 10 imágenes interpolando entre pares de ruidos, y una fila con interpolación **lineal** para comparar. Compare también la norma de los ruidos interpolados con $\sqrt{784}=28$.

In [ ]:
# TODO 3.3 (a) determinismo  (b) slerp e interpolaciones
def slerp(z1, z2, lam):
    raise NotImplementedError

**Respuesta:** _(escriba aquí)_

## Parte 4. Classifier-free guidance (30 min, 5 pts)

### Tarea 4.1 — Etiqueta vacía (0.5 pts)
Implemente `reemplazar_por_vacio(y, p, vacio)`: cada etiqueta se reemplaza por el índice `vacio` (= 10) con probabilidad `p`, de forma independiente. Luego **lance el entrenamiento condicional** (≈4 min).

In [ ]:
def reemplazar_por_vacio(y, p, vacio):
    # TODO 4.1
    raise NotImplementedError

# ---- prueba
y = torch.randint(0, 10, (100000,), device=device)
assert torch.equal(reemplazar_por_vacio(y, 0.0, 10), y) and (reemplazar_por_vacio(y, 1.0, 10) == 10).all()
fr = (reemplazar_por_vacio(y, 0.1, 10) == 10).float().mean().item()
assert abs(fr - 0.1) < 0.01, fr
print(f'OK  fracción vacía con p=0.1: {fr:.3f}')

torch.manual_seed(0)
unet_c = UNet(n_clases=10).to(device)
unet_c_ema = entrenar(unet_c, h_lin, EPOCAS, cond=True, p_vacio=0.1)

### Tarea 4.2 — El ruido guiado (1 pt)
Implemente la ec. 29: $\tilde{\boldsymbol\epsilon}=(1+w)\,\boldsymbol\epsilon_\theta(\mathbf{x}_t,t,c)-w\,\boldsymbol\epsilon_\theta(\mathbf{x}_t,t,\varnothing)$. Para ahorrar tiempo, evalúe ambos términos en **una sola** llamada a la red concatenando el lote consigo mismo.

In [ ]:
def eps_guiado(modelo, x, t, y, w, vacio=10):
    # TODO 4.2
    raise NotImplementedError

# ---- prueba: w = 0 es el modelo condicional; w = -1 es el incondicional
xx, tt = torch.randn(4, 1, 28, 28, device=device), torch.tensor([10, 200, 500, 900], device=device)
yy = torch.tensor([1, 3, 5, 7], device=device)
with torch.no_grad():
    assert torch.allclose(eps_guiado(unet_c_ema, xx, tt, yy, 0.0), unet_c_ema(xx, tt, y=yy), atol=1e-3)
    assert torch.allclose(eps_guiado(unet_c_ema, xx, tt, yy, -1.0), unet_c_ema(xx, tt, y=torch.full_like(yy, 10)), atol=1e-3)
print('OK')

### Tarea 4.3 — La curva fidelidad–diversidad (2 pts)
Para $w\in\{0,\,0.5,\,1,\,2,\,4,\,8\}$ genere 1000 imágenes (100 de cada clase, en el orden de `y_pedido`) con DDIM ($\eta=0$, 50 pasos) **desde el mismo ruido** y calcule:
* **exactitud de clase**: fracción de muestras que el clasificador auxiliar asigna a la clase pedida (fidelidad);
* **diversidad intra-clase** relativa: `diversidad_por_clase(f, y_pedido) / DIV_REAL`;
* **FID\*** contra `f_reales`.

Grafique las tres métricas contra $w$ y la diversidad contra la exactitud (una curva con los puntos etiquetados por $w$). Muestre una cuadrícula (filas = clases, 8 columnas) para $w=0$, $w=2$ y $w=8$. Copie los valores en la Tabla 2 de las instrucciones.

In [ ]:
modelo_c = rapido(unet_c_ema)
W = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
y_pedido = torch.arange(10).repeat_interleave(N_FID // 10)
torch.manual_seed(7)
xT_fijo = torch.randn(N_FID, 1, 28, 28)
res, ejemplos = [], {}
# TODO 4.3: para cada w, generar por lotes de 500 con
#   muestrear_ddim(lambda xx, tt: eps_guiado(modelo_c, xx, tt, yb, w), h_lin, zb.shape, pasos=50, xT=zb)
#   donde yb = y_pedido[i:i+500] y zb = xT_fijo[i:i+500] (en el dispositivo)
# calcular exactitud, diversidad relativa y FID*; guardar en res; graficar

### Tarea 4.4 — Análisis (1.5 pts)
Responda en no más de media página: (a) ¿qué $w$ minimiza el FID\*? ¿Por qué el FID\* empeora para $w$ grande aunque la exactitud siga subiendo? (b) Relacione la curva diversidad–exactitud con el muestreo de $p(\mathbf{x})\,p(c\mid\mathbf{x})^{1+w}$ (notas §8). (c) ¿Qué artefactos ve con $w=8$? ¿Qué pasaría con $p_{\text{vacío}}=0$ en el entrenamiento?

**Respuesta:** _(escriba aquí)_

## Parte 5. Preguntas de cierre (10 min, 2 pts)

Responda brevemente (3–5 líneas cada una), **usando sus propios números**:
1. Compare el modelo de difusión con la GAN de la Clase 11 en tres ejes: estabilidad del entrenamiento (¿cómo se veían las curvas de pérdida?), cobertura de modos (Tarea 2.3 vs. G4) y costo de generar una imagen (evaluaciones de la red: Tarea 3.2 vs. una pasada de $G$).
2. ¿Por qué pudo usar DDIM (Parte 3) sin reentrenar el modelo entrenado con el Algoritmo 1? Responda con una frase sobre qué depende $L_{\text{simple}}$.
3. Stable Diffusion hace todo esto en un espacio latente de $64\times64\times4$. Con sus tiempos de la Tarea 3.2, estime cuánto más lento sería muestrear directamente en $512\times512\times3$ si el costo de la U-Net creciera proporcionalmente al número de valores.

**Respuesta:** _(escriba aquí)_